# FSCI 396: Regression

------------------
Welcome to today's tutorial! 
Today we'll use Python in Jupyter Notebook to perform some common statistical analyses.
We will cover: Regression

The goal is to see how we can analyze data with Python, just like we discussed in lectures.


### Steps to Debugging 

I am here to help you! But before you contact me, make sure you have tried the following:
1. Ensure that all brackets and parentheses are paired.
2. Ensure that your code does not have any typos (eg. when calling your data file).
3. Ensure you did not add additional spaces (eg. when calling your data file).
4. You have restarted your kernel and reun your code. 
5. You have tried to understand the error message. 

If you have done all of these things, I am very happy to help! 

In [ ]:
# Import Modules
# ---------------

# Modules in Python are like library books. 
# Each book contains a set of instructions or “recipes” for doing specific tasks.
# - pandas: for handling and analyzing tables of data
# - seaborn & matplotlib: for making plots
# - numpy: for numerical calculations
# - statsmodels: for running statistical models and regression analysis
# - statsmodels.formula.api (smf): convenient formula interface
# - statsmodels.api (sm): tools for more advanced statistical modeling

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.anova import anova_lm

# Ignore Warnings for simplicity
import warnings
warnings.filterwarnings('ignore')

# Quick check to see if modules loaded correctly
print("Modules loaded successfully!")


In [ ]:
# Load datafile/dataframe (df) from CSV
# --------------------------------------

# Here we load our data. Make sure the CSV file is in the same folder as this notebook.

# If your data is .csv, use these two lines: 
filename = 'SampleData.csv'    # Name of your file
df = pd.read_csv(filename)         # Read the CSV into a DataFrame

# Checkpoint to ensure the file has been loaded 
print(filename, 'has been loaded')


# Regression


- **Regression** allows you to predict Y from X and quantify the strength and direction of their relationship.  
    *Can be extended to multiple X variables (multiple regression) or nonlinear relationships.* \
    *For the purpose of this course, we will focus on mutliple linear regression.*
    
- **Interpretation**: look at the coefficients for direction/size of effects, p-values for significance, and R² for overall model fit. 
  - *"coef" --> estimated effect of the predictor on the outcome.* 
  - *"std err" --> uncertainty of each estimate.* 
  - *"t" --> tests whether each coefficient is significantly different from the null hypothesis in units of standard errors”.* 
  - *"P>|t|" --> the p value associated with the t-statistic (t). Low value means model is statistically significant.* 
  - *"R-squared"  --> how well the model explains the data. Between 0 and 1. Closer to 1 means model explains more of the outcome variation.* 
  - *"Adj. R-squared" --> Like R-squared but penalizes for adding unnecessary variables.*
  - *"F-statistic" --> tests whether at least one predictor is meaningful.*   
  - *"Prob (F-statistic)" --> tests the significance of the F-statistic. Low value means model is statistically significant.*  



- **Quick Translation:**  
    *t and P>|t| --> “Does this specific predictor matter?”* \
    *F and Prob(F) --> “Does the model as a whole matter?”* \
    *R² and Adj. R² --> “How much of the outcome variation does the model explain?”*

In [ ]:
# Define the variable you want to predict

# If all your measures are continuous (eg. continuous number lines), you can follow this:

predicted = 'SelfEfficacy'
# Define the variables you want to feed into the model 
predictors = 'Anxiety_test + Anxiety_writing + Anxiety_speaking ' 

# This puts everything into the model for you 
model = smf.ols(f"{predicted} ~ {predictors}", data=df).fit()

# Print a summary of the defined model 
print(model.summary())

In [ ]:
# If you are working with an categorical measurs, you need to tell python that it is categorical
# For example, we take the same code as above, but this time include gender

predicted = 'SelfEfficacy'
# Define the variables you want to feed into the model 
predictors = 'Anxiety_test + Anxiety_writing + Anxiety_speaking + C(Gender)' 
                                                            # Notice the C and () have been included

# This puts everything into the model for you 
model = smf.ols(f"{predicted} ~ {predictors}", data=df).fit()

# Print a summary of the defined model 
print(model.summary())

# Assumption Checks (BEFORE interpretting regression)

These checks are ideally done interpretting your regression results. However, some of these tests require residuals from doing regression. For this reason, we show you how to do regression first. 
If any of these tests fail, it does*not necessarily mean that regression cannot be performed. It simply means that additional assumption checks or corrections may be needed.  

Here, we include assumption checks for:

- Linearity
- Normality
- Homoscedasticity
- Multicollinearity


###  **Linearity (Residuals vs Fitted Plot)**  
   - Residuals should be scattered randomly around 0 without any clear pattern.  
   - If you see a pattern or a curve, the relationship might not be linear!

In [ ]:
# Extract fitted values and residuals
fitted_vals = model.fittedvalues
residuals = model.resid

# Linearity check (residuals vs fitted)
sns.residplot(x=fitted_vals, y=residuals, lowess=True, line_kws={'color': 'red'})
plt.xlabel('Fitted values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted (Linearity Check)')
plt.show()

###  **Normality of Residuals (Q-Q Plot)**  
   - Points should roughly follow the 45° reference line.  
   - Large deviations at ends indicate that residuals may not be normally distributed.  


In [ ]:
# Normality of residuals (Q-Q plot)
sm.qqplot(residuals, line='45')
plt.title('Q-Q Plot (Normality Check)')
plt.show()

### **Homoscedasticity (Breusch-Pagan Test)**    
- LM Statistic: larger values suggest stronger evidence that residual variance depends on predictors (potential heteroscedasticity).
- LM p-value --> measures the statistical signifiance of the LM statistics. 

**Interpretation**: 
  - LM p-value > 0.05 --> residual variance is roughly constant → assumption satisfied.  
  - LM p-value < 0.05 --> residual variance depends on predictors → heteroscedasticity present.  

In [ ]:
# Homoscedasticity (Breusch-Pagan test)
X_const = sm.add_constant(df[['Rehearsal', 'Elaboration']].dropna())
bp_test = het_breuschpagan(residuals, X_const)
print("Breusch-Pagan test (homoscedasticity):")
print(f"LM Statistic: {bp_test[0]:.4f},\n LM p-value: {bp_test[1]:.4f}")


### Variance Inflation Factor (VIF)
- Measures how strongly each predictor is correlated with the others.  
- VIF ≈ 1 → low correlation (good).  
- VIF > 5–10 → high correlation, predictor may be redundant.  


In [ ]:
# Multicollinearity (VIF)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_const.values, i) for i in range(X_const.shape[1])]
print("Variance Inflation Factor (multicollinearity check):")
print(vif_data)